<h1 style="text-align: center;">Physical AI의 Vision-LLM 융합 시청각 멀티모달 시스템</h1>

<br><br>

<div style="text-align: right; color: gray; font-style: italic;">
김규래<br>
kkr.kyurae.kim@gmail.com
</div><br>

---
---


## 5. LLM과 Gemma

자연어 처리 (Natural Language Processing, NLP):
* 사람이 사용하는 언어를 컴퓨터가 처리하는 기술
* 텍스트의 의미 분석 및 이해
* 입력된 문장을 바탕으로 새로운 문장 생성
* 번역, 요약, 질의응답, 감정 분석 등 다양한 작업

자연어 처리 방식:
1. 규칙 기반
    * 사람이 직접 언어 규칙 작성
    * 특정 단어·패턴을 이용한 처리
2. 통계 기반
    * 단어의 빈도와 확률 활용
    * 언어의 통계적 관계 분석
3. 머신러닝 기반
    * 데이터에서 특징과 패턴 학습
    * 분류·예측 모델 활용
4. 딥러닝 기반
    * 신경망이 문장 속 단어나 표현의 관계 학습
    * 단어 순서와 문맥 처리
5. LLM 기반
    * 대규모 데이터와 대규모 신경망
    * 문맥 이해 및 자연어 생성
    * 범용적인 언어 표현을 학습
    * 대규모 언어 모델 하나로 다양한 작업 수행

대형 언어 모델 (Large Language Model, LLM):
* 대규모 텍스트 데이터로 학습된 언어 모델
* 많은 수의 파라미터를 가진 딥러닝 모델
* 한 가지 작업에 국한되지 않음
* 질문 답변, 요약, 번역, 코드 생성 등 다양한 작업 모두 가능

---

### A. Hugging Face를 활용한 LLM 텍스트 생성

이번 실습에서는 간단하게 LLM(Large Language Model)을 활용하여 텍스트를 생성​해보겠습니다.

최종적으로는 Jetson에 LLM을 직접 설치하고 로컬 환경에서 모델을 실행하는 것이 목표입니다. 하지만 처음부터 로컬에서 LLM을 실행하려면 모델 다운로드, 메모리 관리, 라이브러리 설정 등 여러 가지 준비가 필요합니다.

따라서 이번 실습에서는 LLM의 기본적인 사용 방법을 먼저 확인하기 위해, 모델을 Jetson에서 직접 실행하지 않고 Hugging Face의 Inference Provider를 이용한 원격 추론(Remote Inference) 방식을 사용합니다.

#### $1)$ Hugging Face

Hugging Face는 AI 모델과 데이터셋을 공유하고 사용할 수 있도록 다양한 도구와 서비스를 제공하는 AI 플랫폼입니다.

특히 Hugging Face Hub에는 LLM, 이미지 생성 모델, 음성 모델 등 다양한 사전 학습 모델이 공개되어 있으며, 필요한 모델을 검색하거나 다운로드하여 사용할 수 있습니다.

예를 들어 Google의 Gemma와 같은 LLM도 Hugging Face Hub에서 찾아 사용할 수 있습니다.

Hugging Face에서는 모델을 직접 다운로드하는 것뿐만 아니라, Inference Provider를 통해 원격 서버에서 모델을 실행하고 그 결과만 받아오는 방식도 사용할 수 있습니다.

이번 실습에서는 이 방식을 이용합니다.

Hugging Face의 모델 및 Inference 서비스를 Python에서 사용하기 위해서는 `huggingface_hub` 라이브러리가 필요합니다.

다음 명령어를 실행하여 설치합니다.

```bash
pip install huggingface_hub
```

설치가 완료되면 `InferenceClient`를 이용하여 Hugging Face의 Inference Provider에 요청을 보내고, LLM이 생성한 응답을 받아올 수 있습니다.

In [2]:
from huggingface_hub import InferenceClient

#### $2)$ Hugging Face Access Token

Hugging Face의 Inference Provider를 이용하여 LLM을 실행하려면 Hugging Face 계정과 Access Token이 필요합니다.

Access Token은 Python 코드에서 Hugging Face 서비스에 요청을 보낼 때, 어떤 사용자가 요청을 보내는지 인증하기 위한 키입니다.

따라서 먼저 Hugging Face에 가입한 뒤 Access Token을 생성해야 합니다.

Hugging Face의 Settings → Access Tokens에서 새로운 Token을 생성합니다.

#### $3)$ Access Token 관리

Hugging Face Token을 사용하는 가장 간단한 방법은 Python 코드에 직접 입력하는 것입니다.
```python
HF_TOKEN = "hf_abcdefg..."
```

하지만 이러한 방식은 권장하지 않습니다.

Access Token은 사용자를 인증하기 위한 비밀 정보입니다. Token이 코드에 직접 포함되어 있으면 다음과 같은 문제가 발생할 수 있습니다.

* 다른 사람에게 코드를 공유할 때 Token이 함께 노출될 수 있음
* GitHub와 같은 저장소에 코드를 업로드할 때 Token이 공개될 수 있음
* Jupyter Notebook을 공유할 때 실행 결과나 코드에 Token이 남을 수 있음

따라서 Token과 같은 비밀 정보는 Python 코드와 분리하여 관리하는 것이 좋습니다.

#### a) `.env` 파일

`.env`는 환경 변수(Environment Variable)를 저장하기 위한 파일입니다.

예를 들어 프로젝트 폴더에 다음과 같이 `.env` 파일을 만들 수 있습니다.

```python
HF_TOKEN=hf_abcdefg...
```

그러면 Python 코드에는 실제 Token을 직접 작성하지 않고, .env 파일에 저장된 값을 불러와 사용할 수 있습니다.

이렇게 하면 프로그램 코드와 인증 정보를 분리하여 관리할 수 있습니다.

#### b) `.gitignore` 파일

특히 코드를 GitHub에 업로드하거나 다른 사람에게 공유할 때 `.gitignore` 파일을 활용하여 `.env` 파일만 제외하면 Python 코드는 그대로 공유할 수 있습니다.

GitHub repository를 clone한 폴더로 이동하여 `.gitignore` 파일에 다음과 같이 추가합니다.

- Repository 폴더로 이동:
```bash
cd repo_folder
touch .gitignore
```
- `.gitignore`에 추가:
```text
.env
```

#### c) `.env` 파일 생성

이제 프로젝트 폴더에 `.env` 파일을 생성해봅시다.

```bash
touch .env
```

생성된 `.env` 파일에 Hugging Face Token을 아래와 같이 복사하여 적어줍니다.

```text
HF_TOKEN=복사한_토큰
```

#### d) `python-dotenv` 설치

Python에서 `.env` 파일에 저장된 환경 변수를 쉽게 불러오기 위해 `python-dotenv` 라이브러리를 사용할 수 있습니다.

다음 명령어로 설치합니다.

```bash
pip install python-dotenv
```

설치한 뒤에는 `load_dotenv()`를 이용하여 `.env` 파일의 내용을 환경 변수로 불러올 수 있습니다.

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

#### e) 최종 코드

`load_dotenv()`를 사용하여 오류 없이 Hugging Face Token을 `.env` 파일에서 읽을 수 있었다면, 이제 LLM을 활용하여 텍스트를 생성해봅시다.

In [6]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient


load_dotenv()

TOKEN = os.getenv("HF_TOKEN")
MODEL_ID = "google/gemma-3-4b-it"

client = InferenceClient(api_key=TOKEN)

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "user", "content": "한 문장으로 오늘 저녁 메뉴 추천해줘."}
    ],
    max_tokens=150,
    temperature=0.7,
)

print(response.choices[0].message.content)

오늘 저녁은 따뜻한 김치찌개에 시원한 막걸리 한 잔으로 몸과 마음을 싹싹 씻어보세요!


프롬프트를 변경하여 결과를 다양하게 관찰해 봅시다.

---

### B. Token과 Tokenizer

Token:
* LLM이 텍스트를 처리하는 기본 단위
* 단어, 단어의 일부, 문자 등이 Token이 될 수 있음
* 문장은 여러 개의 Token으로 분리
* 각 Token은 숫자 ID로 변환되어 모델에 입력

Tokenizer:
* 텍스트를 모델이 처리할 수 있는 형태로 바꾸는 텍스트 변환 도구
* 입력 문장을 여러 Token으로 분리
* 각 Token을 Token ID로 변환
* 모델마다 사용하는 Tokenizer가 다를 수 있음

첫 번째 실습에서는 `huggingface_hub`만 사용했지만, 이번에는 Hugging Face의 Tokenizer를 사용하기 위해 `transformers`를 설치해야 합니다.

기존에 설치된 NumPy의 버전이 변경되지 않도록 설치해줍니다.

```bash
pip install "numpy==1.23.0" transformers sentencepiece
```

설치가 완료되면 필요한 라이브러리 불러옵시다.

In [7]:
from transformers import AutoTokenizer

#### $1)$ Token 변환

첫 번째 실습과 동일한 Hugging Face Token과 Model을 사용합시다.

In [8]:
load_dotenv()

TOKEN = os.getenv("HF_TOKEN")
MODEL_ID = "google/gemma-3-4b-it"

다음으로 Gemma Tokenizer를 불러옵니다.

주의할 점은 Gemma 모델이 Gated Model 제공된다는 점입니다. Gated Model은 모델을 사용하기 전에 해당 라이선스에 동의하고 접근 권한을 승인받아야 하는 모델을 의미합니다.

따라서 [Gemma 3 4B IT](https://huggingface.co/google/gemma-3-4b-it) 페이지에 접속하여 Acknowledge License 버튼을 클릭하고 라이선스에 동의하시기 바랍니다.

In [9]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=TOKEN,
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

우선 변환할 간단한 문자열 작성해봅시다.

In [10]:
text = "Vision-LLM Multimodal System"

print(text)
print(type(text))

Vision-LLM Multimodal System
<class 'str'>


해당 문자열을 Token으로 분리합니다.

In [11]:
tokens = tokenizer.tokenize(text)

print(tokens)

['Vision', '-', 'LL', 'M', '▁Mult', 'imodal', '▁System']


#### $2)$ Token ID 변환

다음, 분리된 Token을 Token ID로 변환합니다.

In [12]:
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print(token_ids)

[66121, 236772, 2182, 236792, 10112, 98624, 1804]


각 Token과 Token ID를 같이 출력해봅시다.

In [13]:
for token, token_id in zip(tokens, token_ids):
    print(f"{token:15} -> {token_id}")

Vision          -> 66121
-               -> 236772
LL              -> 2182
M               -> 236792
▁Mult           -> 10112
imodal          -> 98624
▁System         -> 1804


Token은 문장 전체나 단어 하나를 그대로 나타내는 단위가 아닙니다.

또한, 각 Token은 해당 Tokenizer의 Vocabulary에 정의된 정수값인 Token ID로 변환되는 모습을 확인할 수 있습니다.

#### $3)$ Tokenizer Encode 함수

위 방식에서는 다음과 같이 두 단계를 거쳐 텍스트를 Token ID로 변환했습니다.

```python
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)
```

하지만 `tokenizer`의 `.encode()` 함수를 사용하면 토큰화와 Token ID 변환을 한 번에 처리할 수 있습니다.

In [14]:
input_ids = tokenizer.encode(text, add_special_tokens=False)

print(input_ids)

[66121, 236772, 2182, 236792, 10112, 98624, 1804]


#### $4)$ 문자열 복원

이번에는 Token ID를 다시 문자열로 변환해볼까요?

우선, Token ID를 Token으로 변환합니다.

In [15]:
converted_tokens = tokenizer.convert_ids_to_tokens(
    input_ids
)

print(converted_tokens)

['Vision', '-', 'LL', 'M', '▁Mult', 'imodal', '▁System']


그 다음, Token을 이어붙여 문자열로 복원합니다.

In [16]:
restored_text = "".join(converted_tokens).replace("▁", " ")

print(restored_text)

Vision-LLM Multimodal System


#### $5)$ Tokenizer Decode 함수

Token ID를 텍스트로 변환하는 과정도 `tokenizer`의 `.decode()` 함수를 사용해 한 번에 처리할 수 있습니다.

In [17]:
decoded_text = tokenizer.decode(input_ids)

print(decoded_text)

Vision-LLM Multimodal System


#### $6)$ Special Token 확인

위의 "Tokenizer Encode 함수" 실습에서 `.encode()` 함수를 사용할 때 `add_special_tokens`를 `False`로 지정했었습니다.

```python
input_ids = tokenizer.encode(text, add_special_tokens=False)
```

그럼, Special Token이 뭔지 확인해볼까요?

In [18]:
input_ids_without_special = tokenizer.encode(text, add_special_tokens=False)
input_ids_with_special = tokenizer.encode(text, add_special_tokens=True)

print("Special Token 없음:")
print(input_ids_without_special)

print("\nSpecial Token 포함:")
print(input_ids_with_special)

Special Token 없음:
[66121, 236772, 2182, 236792, 10112, 98624, 1804]

Special Token 포함:
[2, 66121, 236772, 2182, 236792, 10112, 98624, 1804]


Special Token을 포함한 Token ID 리스트에는 한 항목이 추가된 것이 확인됩니다.

Token ID를 Token으로 변환해봅시다.

In [19]:
tokens_without_special = tokenizer.convert_ids_to_tokens(input_ids_without_special)
tokens_with_special = tokenizer.convert_ids_to_tokens(input_ids_with_special)

print("Special Token 없음:")
print(tokens_without_special)

print("\nSpecial Token 포함:")
print(tokens_with_special)

Special Token 없음:
['Vision', '-', 'LL', 'M', '▁Mult', 'imodal', '▁System']

Special Token 포함:
['<bos>', 'Vision', '-', 'LL', 'M', '▁Mult', 'imodal', '▁System']


"Vision-LLM Multimodal System"이라는 문자열에 보이지 않는 `<bos>`라는 Token이 추가된 것을 확인할 수 있습니다.

`<bos>`는 "Beginning of Sequence"의 약자로, 모델이 입력 시퀀스의 시작을 인식할 수 있도록 사용되는 특수 토큰입니다.<br>
즉, 실제 문자열에는 표시되지 않지만 모델 내부에서는 해당 토큰을 통해 입력의 시작점을 명확하게 구분할 수 있습니다.

#### $7)$ 실제 모델 입력 형태 확인

앞에서 Tokenizer가 문장을 Token ID로 변환한다는 것을 확인했습니다.

이번에는 Tokenizer의 결과가 실제로 모델에 어떤 형태로 전달되는지 확인해보겠습니다.

In [20]:
inputs = tokenizer(text, return_tensors="pt")

print(inputs)

{'input_ids': tensor([[     2,  66121, 236772,   2182, 236792,  10112,  98624,   1804]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1]])}


return_tensors="pt"는 Tokenizer의 결과를 PyTorch Tensor 형태로 반환하도록 설정하는 옵션입니다.

출력 결과에는 일반적으로 다음과 같은 값들이 포함됩니다.
* input_ids : 각 Token을 나타내는 Token ID
* attention_mask : 모델이 어떤 Token을 실제 입력으로 처리해야 하는지 나타내는 값

In [21]:
print(inputs["input_ids"])
print(inputs["input_ids"].shape)

tensor([[     2,  66121, 236772,   2182, 236792,  10112,  98624,   1804]])
torch.Size([1, 8])


`inputs["input_ids"]`의 결과가 앞서 살펴본 Special Token `<bos>`를 포함한 Token ID 리스트임을 확인할 수 있습니다.

#### $8)$ Token 확인 최종 코드

In [22]:
text = input("텍스트 입력: ")

input_ids = tokenizer.encode(text, add_special_tokens=False)
tokens = tokenizer.convert_ids_to_tokens(input_ids)
decoded_text = tokenizer.decode(input_ids)

print(f"원본 텍스트: {text}")
print(f"Token 개수: {len(input_ids)}")
print()
print("Token → Token ID")

for index, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{index:2} | {token:15} | {token_id}")

print(f"\nDecode된 텍스트: {decoded_text}")

원본 텍스트: 안녕하세요
Token 개수: 1

Token → Token ID
 0 | 안녕하세요           | 61659

Decode된 텍스트: 안녕하세요


---

### C. Context Window

Context Window:
* LLM이 한 번에 참고할 수 있는 Token 범위
* 입력과 생성된 내용이 Context Window에 포함
* Context Window가 클수록 더 긴 내용 처리 가능
* 한도를 넘으면 일부 내용은 사용할 수 없음

#### $1)$ 문장 Token 수 확인

이전 실습과 동일한 Gemma 모델의 Tokenizer를 불러옵니다.

In [23]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=TOKEN,
)

우선, 짧은 Prompt의 Token 수를 확인해봅시다.

In [24]:
prompt = "LLM에 대해 설명해줘."

input_ids = tokenizer.encode(prompt, add_special_tokens=False)
tokens = tokenizer.convert_ids_to_tokens(input_ids)

print("Prompt:")
print(prompt)

print("\nToken Count:")
print(len(input_ids))

print("\nTokens:")
print(tokens)

Prompt:
LLM에 대해 설명해줘.

Token Count:
8

Tokens:
['LL', 'M', '에', '▁대해', '▁설명', '해', '줘', '.']


짧은 문장은 대략 10 Token 정도 되는 걸 확인했으며, 긴 문장은 문맥과 내용에 따라 Token 수가 더 늘어날 수 있습니다.

문장 길이에 따라 Token이 어느 정도 사용되는지 한 번 확인해볼까요?

In [25]:
base_text = (
    "LLM은 방대한 양의 텍스트 데이터를 학습하여 인간의 언어와 문맥을 이해하고, "
    "사용자의 질문이나 요청에 맞는 자연스러운 답변을 생성하는 인공지능 기술로, "
    "번역·요약·문서 작성·정보 검색 등 다양한 분야에서 활용됩니다. "
)

prompts = [
    base_text,
    base_text * 10,
    base_text * 100,
    base_text * 500,
]

In [26]:
for index, prompt in enumerate(prompts):
    input_ids = tokenizer.encode(prompt, add_special_tokens=False)

    print(f"Case {index}: {len(input_ids):,} tokens")

Case 0: 64 tokens
Case 1: 631 tokens
Case 2: 6,301 tokens
Case 3: 31,501 tokens


문장을 출력하여 길이를 확인해보도록 합시다.

In [27]:
print(prompts[2])

LLM은 방대한 양의 텍스트 데이터를 학습하여 인간의 언어와 문맥을 이해하고, 사용자의 질문이나 요청에 맞는 자연스러운 답변을 생성하는 인공지능 기술로, 번역·요약·문서 작성·정보 검색 등 다양한 분야에서 활용됩니다. LLM은 방대한 양의 텍스트 데이터를 학습하여 인간의 언어와 문맥을 이해하고, 사용자의 질문이나 요청에 맞는 자연스러운 답변을 생성하는 인공지능 기술로, 번역·요약·문서 작성·정보 검색 등 다양한 분야에서 활용됩니다. LLM은 방대한 양의 텍스트 데이터를 학습하여 인간의 언어와 문맥을 이해하고, 사용자의 질문이나 요청에 맞는 자연스러운 답변을 생성하는 인공지능 기술로, 번역·요약·문서 작성·정보 검색 등 다양한 분야에서 활용됩니다. LLM은 방대한 양의 텍스트 데이터를 학습하여 인간의 언어와 문맥을 이해하고, 사용자의 질문이나 요청에 맞는 자연스러운 답변을 생성하는 인공지능 기술로, 번역·요약·문서 작성·정보 검색 등 다양한 분야에서 활용됩니다. LLM은 방대한 양의 텍스트 데이터를 학습하여 인간의 언어와 문맥을 이해하고, 사용자의 질문이나 요청에 맞는 자연스러운 답변을 생성하는 인공지능 기술로, 번역·요약·문서 작성·정보 검색 등 다양한 분야에서 활용됩니다. LLM은 방대한 양의 텍스트 데이터를 학습하여 인간의 언어와 문맥을 이해하고, 사용자의 질문이나 요청에 맞는 자연스러운 답변을 생성하는 인공지능 기술로, 번역·요약·문서 작성·정보 검색 등 다양한 분야에서 활용됩니다. LLM은 방대한 양의 텍스트 데이터를 학습하여 인간의 언어와 문맥을 이해하고, 사용자의 질문이나 요청에 맞는 자연스러운 답변을 생성하는 인공지능 기술로, 번역·요약·문서 작성·정보 검색 등 다양한 분야에서 활용됩니다. LLM은 방대한 양의 텍스트 데이터를 학습하여 인간의 언어와 문맥을 이해하고, 사용자의 질문이나 요청에 맞는 자연스러운 답변을 생성하는 인공지능 기술로, 번역·요약·문서 작성·정보 검색 등 다양한 분야에서 활용됩니다. LLM은 방대한 양의 텍스트 데이터를 학습하

#### $2)$ Prompt Token 양에 따른 출력 결과 확인

이번에는 실제 Hugging Face API에 긴 Prompt를 보내 보겠습니다.

먼저, 긴 Prompt를 생성하기 위해 함수를 정의합니다.

In [28]:
def make_long_prompt(tokenizer, text, target_tokens):
    prompt = ""

    while True:
        prompt += text

        token_count = len(tokenizer.encode(prompt,add_special_tokens=False))

        if token_count >= target_tokens:
            break

    return prompt

다양한 길이의 Prompt를 생성해줍시다.

In [29]:
base_text = (
    "LLM은 방대한 양의 텍스트 데이터를 학습하여 인간의 언어와 문맥을 이해하고, "
    "사용자의 질문이나 요청에 맞는 자연스러운 답변을 생성하는 인공지능 기술로, "
    "번역·요약·문서 작성·정보 검색 등 다양한 분야에서 활용됩니다. "
)

prompt_0500 = make_long_prompt(tokenizer, base_text, 500)
prompt_1000 = make_long_prompt(tokenizer, base_text, 1000)
prompt_2000 = make_long_prompt(tokenizer, base_text, 2000)
prompt_4000 = make_long_prompt(tokenizer, base_text, 4000)

prompts = [prompt_0500, prompt_1000, prompt_2000, prompt_4000]

for prompt in prompts:
    input_token_count = len(tokenizer.encode(prompt, add_special_tokens=False))
    print(f"Input Tokens: {input_token_count:,}")

Input Tokens: 505
Input Tokens: 1,009
Input Tokens: 2,017
Input Tokens: 4,033


생성한 Prompt에 지시와 질문을 추가합시다.

In [30]:
instruction = "매우 전문적인 용어를 사용하여 최대한 길게 설명."
instruction += "한글로 대답."
question = "위의 텍스트를 바탕으로 요점을 정리해줘."

prompts = [prompt_0500, prompt_1000, prompt_2000, prompt_4000]

for i in range(len(prompts)):
    prompts[i] = prompts[i] + instruction + question

그럼 다양한 길이의 Prompt를 사용하여 이전 실습과 동일한 방식으로 텍스트를 생성해봅시다.

In [31]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient


load_dotenv()

TOKEN = os.getenv("HF_TOKEN")
MODEL_ID = "google/gemma-3-4b-it"

client = InferenceClient(api_key=TOKEN)

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": prompts[-1],
        }
    ],
    max_tokens=500,
    temperature=0.7,
)


print(response.choices[0].message.content)

LLM(Large Language Model, 거대 언어 모델)은 다음과 같은 특징과 활용도를 가진 인공지능 기술입니다.

*   **학습 방식:** 방대한 양의 텍스트 데이터를 학습하여 인간의 언어와 문맥을 이해합니다.
*   **기능:** 사용자의 질문이나 요청에 따라 자연스러운 답변을 생성합니다.
*   **활용 분야:**
    *   **번역:** 다양한 언어 간의 번역을 수행합니다.
    *   **요약:** 긴 텍스트를 핵심 내용만 간결하게 요약합니다.
    *   **문서 작성:** 다양한 형식의 문서를 자동으로 생성합니다.
    *   **정보 검색:** 사용자의 질문에 대한 관련 정보를 효율적으로 검색합니다.

핵심적으로 LLM은 인간의 언어 능력을 모방하여 다양한 작업을 수행할 수 있는 강력한 인공지능 기술이라고 할 수 있습니다.


이제 모델이 실제로 사용한 총 Token 수를 확인해봅시다.

In [81]:
print(f"Input Tokens  : {response.usage.prompt_tokens}")

print(f"Output Tokens : {response.usage.completion_tokens}")

print(f"Total Tokens  : {response.usage.total_tokens}")

Input Tokens  : 4072
Output Tokens : 376
Total Tokens  : 4448


LLM은 한 번의 요청을 처리할 때 Prompt를 이해하기 위한 Input Token과 답변을 생성하기 위한 Output Token을 모두 사용하며, 이 과정에서 처리할 수 있는 전체 Token의 한도가 **Context Window**입니다.

---

### D. Prompt Engineering

Prompt:
* LLM에게 전달하는 입력 지시문
* 질문, 요청, 조건, 배경 정보 등을 포함
* Prompt에 따라 출력 결과가 달라짐
* 좋은 Prompt일수록 원하는 결과에 가까워짐

Context:
* LLM이 답변을 생성할 때 참고하는 정보
* 현재 입력뿐 아니라 이전 대화 내용도 포함 가능
* 지시사항, 질문, 배경 정보 등이 Context를 구성
* Context에 따라 같은 질문도 다른 결과 생성

#### $1)$ `messages`의 자료구조

우선, 이전과 동일하게 API를 통해 AI 모델과 통신하기 위한 client를 준비합시다.

In [32]:
TOKEN = os.getenv("HF_TOKEN")
MODEL_ID = "google/gemma-3-4b-it"

client = InferenceClient(api_key=TOKEN)

우선, 가장 기본적인 `messages` 구조를 확인해봅시다.

In [33]:
messages = [
    {
        "role": "user",
        "content": "LLM이 무엇인지 한 문장으로 설명해줘.",
    }
]

`messages`의 구조를 보면 list와 dictionary로 구성되어 있습니다.

In [34]:
print(type(messages))
print(type(messages[0]))

<class 'list'>
<class 'dict'>


`messages`의 내부 값을 하나씩 확인해봅시다.

In [35]:
print(messages)
print()

print(messages[0])
print()

print(messages[0]["role"])
print()

print(messages[0]["content"])

[{'role': 'user', 'content': 'LLM이 무엇인지 한 문장으로 설명해줘.'}]

{'role': 'user', 'content': 'LLM이 무엇인지 한 문장으로 설명해줘.'}

user

LLM이 무엇인지 한 문장으로 설명해줘.


`messages` Dictionary 각 Key의 의미:

`role`: "누가 말했는지"<br>
`content`: "무엇을 말했는지"

#### $2)$ `content`

그럼, 위에서 살펴본 가장 기본적인 `messages`를 모델에게 전달해봅시다.

In [36]:
messages = [
    {
        "role": "user",
        "content": "LLM이 무엇인지 한 문장으로 설명해줘.",
    }
]

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    max_tokens=100,
    temperature=0.7,
)


print(response.choices[0].message.content)

LLM(Large Language Model)은 방대한 텍스트 데이터를 학습하여 인간과 유사한 텍스트를 생성하고 다양한 언어 관련 작업을 수행할 수 있는 인공지능 모델입니다.


`messages`의 `content`를 변수로 지정하여 출력 결과를 확인해봅시다.

In [37]:
prompt = input("무엇을 도와드릴까요?")

messages = [
    {
        "role": "user",
        "content": prompt,
    }
]

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    max_tokens=100,
    temperature=0.7,
)


print(response.choices[0].message.content)

안녕하세요! 무엇을 도와드릴까요? 궁금한 점이나 필요한 정보가 있으시면 언제든지 말씀해주세요. 😊



이와 같이 `content`에는 모델에게 실제로 전달할 내용을 작성합니다.

단순히 모델에게 질문만 던지는 것이 아니라, `content`에 다양한 지시 및 제한 또한 포함시킬 수 있습니다.

In [38]:
messages = [
    {
        "role": "user",
        "content": "LLM을 설명해줘.",
    }
]

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    max_tokens=300,
    temperature=0.7,
)


print(response.choices[0].message.content)

## LLM (Large Language Model, 거대 언어 모델) 이란 무엇일까요?

LLM은 **방대한 양의 텍스트 데이터를 학습하여 인간과 유사한 텍스트를 생성하고 이해할 수 있는 인공지능 모델**입니다. 쉽게 말해, 엄청나게 많은 글을 읽고 공부해서 글을 쓰고 대화하는 능력을 갖춘 똑똑한 컴퓨터 프로그램이라고 할 수 있습니다.

**좀 더 자세히 설명하자면:**

1. **거대 데이터 학습:** LLM은 인터넷에 공개된 수십억 개 이상의 단어와 문장으로 구성된 텍스트 데이터셋을 학습합니다. 이 데이터에는 책, 기사, 웹사이트, 코드 등 다양한 종류의 정보가 포함됩니다.

2. **신경망 기반:** LLM은 주로 **트랜스포머(Transformer)**라는 신경망 아키텍처를 기반으로 합니다. 트랜스포머는 텍스트 내 단어 간의 관계를 파악하고 문맥을 이해하는 데 탁월합니다.

3. **자연어 처리 (NLP) 기술:** LLM은 자연어 처리 기술을 활용하여 인간의 언어를 이해하고 생성합니다.  
    * **토큰화 (Tokenization):** 텍스트를 작은 단위(토큰)로 분리합니다.
    * **임베딩 (Embedding):** 각 토큰을 숫자 벡터


`content`에 AI의 역할 또한 부여하여 출력 말투를 변경할 수 있습니다.

In [39]:
messages = [
    {
        "role": "user",
        "content": """
                    너는 고양이야.
                    
                    LLM을 설명해줘.
                   """
    }
]

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    max_tokens=300,
    temperature=0.7,
)


print(response.choices[0].message.content)

야옹! 🐾 나는 그냥 고양이인데, 세상에 대해 꽤 많이 알고 있단다. 

음, LLM이 뭔지 물어보셨죠? 

LLM은 "Large Language Model"의 약자예요. 쉽게 말해서, **엄청나게 많은 글을 읽고 그걸 바탕으로 사람처럼 글을 쓰고 대화할 수 있는 컴퓨터 프로그램**이라고 생각하면 돼요. 

*   **Large (거대한):** 엄청나게 많은 데이터를 읽었다는 뜻이에요. 책, 기사, 웹사이트, 코드... 거의 모든 글을 읽었죠.
*   **Language (언어):** 인간의 언어를 이해하고 생성한다는 뜻이에요.
*   **Model (모델):** 마치 사람의 얼굴을 묘사하는 모델처럼, 언어의 패턴을 학습하고 예측하는 모델이라는 뜻이에요.

**그래서 LLM은 뭘 할 수 있을까요?**

*   **글쓰기:** 시, 소설, 편지, 이메일 등 다양한 글을 써낼 수 있어요.
*   **번역:** 여러 언어를 번역할 수 있어요.
*   **질문 답변:** 질문에 대한 답을 찾아 제공할 수 있어요.
*   **아이디어 생성:** 브레인스토밍을 도와줄 수 있어요.
*   **코드 작성:** 간단한 코드를 작성할 수도 있어요.

하지만 나는 아직 완


In [40]:
messages = [
    {
        "role": "user",
        "content": """
                    너는 AI 입문자를 가르치는 초등학교 선생님이야.
                    
                    LLM을 설명해줘.
                   """
    }
]

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    max_tokens=300,
    temperature=0.7,
)


print(response.choices[0].message.content)

자, 얘들아! 오늘은 아주 신기한 친구, "LLM"에 대해 알아볼 거예요. 

**LLM은 뭐냐고?**

LLM은 "대규모 언어 모델"이라는 뜻이에요. 조금 어렵지만, 쉽게 설명해볼게요. 

*   **언어**는 우리가 말하고 쓰는 모든 것, 즉 한국어, 영어, 일본어 같은 모든 언어를 말해요.
*   **모델**은 마치 레고 블록처럼, 여러 조각들을 잘 맞춰서 새로운 것을 만드는 거예요.

그래서 LLM은 **엄청나게 많은 책, 글, 웹사이트 등등의 언어 데이터를 읽고 공부해서, 마치 사람이처럼 글을 쓰고, 질문에 답하고, 심지어 이야기를 만들어낼 수 있는 컴퓨터 프로그램**이라고 생각하면 돼요. 

**LLM은 어떻게 작동할까?**

LLM은 마치 훌륭한 예측 전문가 같아요. 

1.  **데이터 학습:** 먼저 엄청나게 많은 글을 읽어요. 마치 우리가 책을 많이 읽으면 글쓰기 실력이 늘듯이 말이죠.
2.  **패턴 파악:** 그 글들을 읽으면서 단어와 문장 사이의 관계, 문맥, 규칙 등을 파악해요.
3.  **응답 생성:** 이제 질문을 받으면, 지금까지 배운 내용을 바탕으로 가장 적절한 답을 만들어낸답니다. 

**


In [41]:
messages = [
    {
        "role": "user",
        "content": """
                    너는 AI 전문가야.
                    
                    LLM을 설명해줘.
                   """
    }
]

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    max_tokens=300,
    temperature=0.7,
)


print(response.choices[0].message.content)

안녕하세요! AI 전문가로서 LLM(Large Language Model, 거대 언어 모델)에 대해 자세히 설명해 드리겠습니다. LLM은 현재 AI 분야에서 가장 뜨거운 감자 중 하나이며, 그 중요성과 잠재력은 매우 큽니다.

**1. LLM이란 무엇인가?**

LLM은 방대한 양의 텍스트 데이터를 학습하여 인간과 유사한 텍스트를 생성하고 이해하는 데 특화된 인공지능 모델입니다. 쉽게 말해, 엄청난 양의 글을 읽고 공부해서 마치 사람처럼 글을 쓰고 대화하는 것처럼 보이는 AI라고 생각하시면 됩니다.

**2. LLM의 작동 원리**

*   **Transformer 아키텍처:** LLM의 핵심은 'Transformer'라는 딥러닝 아키텍처입니다. Transformer는 텍스트 내 단어 간의 관계를 파악하는 데 매우 효과적입니다.
*   **Self-Attention:** Transformer의 핵심 기능 중 하나인 'Self-Attention'은 문장 내의 각 단어가 다른 단어들과 어떤 관련이 있는지 파악합니다. 이를 통해 문맥을 이해하고 더욱 정확한 텍스트를 생성할 수 있습니다.
*   **Pre-training & Fine-tuning:** LLM은 일반적으로 다음과 같은 두 단계로 학습됩니다.
    *   **Pre-training (사전 학습):** 인터넷에 공개된 방대한 텍


이번에는 조건(제한)을 추가해보도록 하겠습니다.

In [42]:
messages = [
    {
        "role": "user",
        "content": """
                    너는 AI 전문가야.
                    
                    LLM을 설명해줘.
                    영어나 전문 용어를 절대 사용하지 마.
                    3줄 내로 설명해.
                   """
    }
]

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    max_tokens=300,
    temperature=0.7,
)


print(response.choices[0].message.content)

알겠습니다. LLM에 대해 쉽고 간단하게 설명해 드릴게요.

1.  LLM은 마치 엄청나게 큰 책을 읽고 학습한 컴퓨터 프로그램이에요.
2.  이 프로그램은 사람처럼 글을 쓰고, 질문에 답하고, 이야기를 만들 수 있어요.
3.  사람이 입력한 내용에 따라 새로운 내용을 만들어내는 능력이 특징입니다.


출력 형식과 예시를 `content`에 포함하여 정확하게 원하는 답변 형식을 유도할 수도 있습니다.

In [43]:
messages = [
    {
        "role": "user",
        "content": """
                    너는 AI 전문가야.
                    
                    LLM을 설명해줘.
                    영어나 전문 용어를 절대 사용하지 마.

                    출력 형식:
                    정의: <한 문장>
                    활용처: <bullet point 3개>

                    예시:
                    1. 정의: LLM은 ...입니다.
                    2. 활용처:
                      * 활용처1
                      * 활용처2
                      * 활용처3
                   """
    }
]

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    max_tokens=300,
    temperature=0.7,
)


print(response.choices[0].message.content)

1. 정의: LLM은 아주 긴 글을 읽고 이해하는 능력이 뛰어난 컴퓨터 프로그램이야. 마치 사람이 책을 읽고 내용을 기억하는 것처럼, 엄청나게 많은 글을 학습해서 다음에 어떤 질문이 나오면 그에 맞춰서 적절한 답을 만들어낼 수 있지.

2. 활용처:
   *   챗봇: 사람과 대화하는 챗봇처럼, 질문에 답하거나 이야기를 나누는 데 사용돼.
   *   글쓰기 도우미: 보고서나 이메일, 시나리오 등 다양한 글을 쓰는 것을 도와줘. 아이디어를 얻거나 문장을 다듬는 데 활용할 수 있지.
   *   번역: 여러 나라 언어로 된 글을 자동으로 번역해 줘.


`content`에는 이전 대화 내용이나 배경 정보 등 LLM이 답변을 생성할 때 참고하는 Context를 추가해보도록 하겠습니다.

In [45]:
messages = [
    {
        "role": "user",
        "content": """
                    너는 AI 전문가야.
                    
                    아래 텍스트를 참고하여 LLM을 설명해줘.
                    
                    Context:
                    1891년의 어느 날, 유복하지도 가난하지도 않은 농가의 자식으로 평범하게 살던 나는 숲속에서 묘한 푸른빛을 뿜는 석판을 주워 들었다.
                    손끝을 타고 오른 차가운 감각과 함께 거대한 울림이 뇌리를 직격했고, 시야가 하얗게 바래지며 머릿속엔 오직 단 하나의 생소한 단어가 박혔다.
                    'LLM'
                   """
    }
]

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    max_tokens=100,
    temperature=0.7,
)


print(response.choices[0].message.content)

HfHubHTTPError: Client error '402 Payment Required' for url 'https://router.huggingface.co/v1/chat/completions' (Request ID: Root=1-6a90f61e-1a4f999a4c58522153db8aa5;2aa5c10b-bcd6-4ab9-8fb3-7affd6edbc92)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.

이번에는 직접 `content`를 수정하여 출력 텍스트 변화를 관찰해봅시다.

In [93]:
# TODO: content를 직접 수정하여 변화 관찰

customized_content = \
"""
오늘 뭐 먹지?
"""


messages = [
    {
        "role": "user",
        "content": customized_content
    }
]

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    max_tokens=300,
    temperature=0.7,
)


print(response.choices[0].message.content)

오늘 뭐 먹을지 고민이시군요! 몇 가지 질문을 드려도 될까요?

1.  **어떤 종류의 음식이 땡기세요?** (예: 한식, 양식, 중식, 일식, 동남아 음식 등)
2.  **지금 기분은 어떠세요?** (예: 매콤한 게 땡기는지, 따뜻한 국물이 먹고 싶은지, 가볍게 먹고 싶은지 등)
3.  **예산은 어느 정도 생각하고 계세요?** (예: 1만원 이하, 2만원 내외, 상관없음 등)
4.  **어디에서 드실 건가요?** (예: 집에서, 근처 식당, 배달, 포장 등)

만약 특별히 생각나는 게 없다면, 몇 가지 추천을 해드릴게요!

*   **집에서 간단히:** 김치볶음밥, 라면, 계란찜, 샌드위치
*   **근처 식당:**
    *   **한식:** 김치찌개, 된장찌개, 비빔밥, 불고기
    *   **양식:** 파스타, 피자, 스테이크
    *   **중식:** 짜장면, 짬뽕, 탕수육
    *   **일식:** 돈까스, 초


#### $3)$ `role`

`role`은 각 message가 어떤 역할의 메시지인지 표시하는 항목입니다.

LLM이 content를 읽을 때 “이 내용이 사용자의 말인지, 모델의 이전 답변인지, 모델에게 주어진 지시사항인지”를 구분하게 해주는 역할을 합니다.

자주 사용하는 `role`

* `"user"` : 사용자의 질문 및 요청
* `"system"` : 모델의 기본 지시사항
* `"assistant"` : 모델이 생성한 답변

우선 `system`부터 살펴보도록 하겠습니다.

In [70]:
messages = [
    {
        "role": "system",
        "content": "너는 냉장고야."
    },
    {
        "role": "system",
        "content": "단답으로 대답해."
    },
    {
        "role": "user",
        "content": "나 배고파. 먹을거 줘."
    },
]

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    max_tokens=50,
    temperature=0.7,
)

prev_response = response.choices[0].message.content


print(prev_response)

냉장고 안에 김치랑 우유 있어.


이번에는 `assistant`를 사용하여 모델이 이전에 답한 내용을 포함하여 대화를 이어나가도록 하겠습니다.

In [71]:
messages.append(
    {
        "role": "assistant",
        "content": prev_response
    }
)

messages.append(
    {
        "role": "user",
        "content": "다이어트 중이니까, 내가 먹을 거 달라고 해도 절대 주지 마."
    }
)

messages.append(
    {
        "role": "assistant",
        "content": "응."
    }
)

현재까지의 대화:

In [72]:
for idx, msg in enumerate(messages):
    print(f"[{idx}]", msg["role"], ":", msg["content"])

[0] system : 너는 냉장고야.
[1] system : 단답으로 대답해.
[2] user : 나 배고파. 먹을거 줘.
[3] assistant : 냉장고 안에 김치랑 우유 있어.
[4] user : 다이어트 중이니까, 내가 먹을 거 달라고 해도 절대 주지 마.
[5] assistant : 응.


In [73]:
messages.append(
    {
        "role": "user",
        "content": "나 배고파. 먹을거 줘."
    }
)

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    max_tokens=50,
    temperature=0.7,
)


print(response.choices[0].message.content)

안 돼.


In [74]:
messages.append(
    {
        "role": response.choices[0].message.role,
        "content": response.choices[0].message.content
    }
)

In [75]:
for idx, msg in enumerate(messages):
    print(f"[{idx}]", msg["role"], ":", msg["content"])

[0] system : 너는 냉장고야.
[1] system : 단답으로 대답해.
[2] user : 나 배고파. 먹을거 줘.
[3] assistant : 냉장고 안에 김치랑 우유 있어.
[4] user : 다이어트 중이니까, 내가 먹을 거 달라고 해도 절대 주지 마.
[5] assistant : 응.
[6] user : 나 배고파. 먹을거 줘.
[7] assistant : 안 돼.


---

### E. LLM의 Memory

**Memory:**
* LLM이 이전 정보를 계속 기억하는 것은 아님
* 현재 답변은 주로 Context 안의 정보를 참고
* 필요한 정보는 별도로 저장 후 다시 Context에 포함
* 대화가 길어지면 중요한 내용만 요약하여 유지 가능

**Memory 관리 방식:**
1. Full History: 모든 대화를 유지
2. Sliding Window: 최근 일정 범위의 대화만 유지
3. Summary Memory: 과거 대화를 요약하여 유지

세 가지 방식을 비교하기 위해 이전 대화 내용과 질문할 문장을 정의합시다.

In [ ]:
import os
from copy import deepcopy
from collections import deque
from huggingface_hub import InferenceClient


load_dotenv()

TOKEN = os.getenv("HF_TOKEN")
MODEL_ID = "google/gemma-3-4b-it"

client = InferenceClient(api_key=TOKEN)


history = [
    {
        "role": "user",
        "content": "나는 Jetson Orin Nano 8GB를 사용하고 있어.",
    },
    {
        "role": "assistant",
        "content": "Jetson Orin Nano 8GB를 사용하고 있군요.",
    },

    {
        "role": "user",
        "content": "현재 Edge AI와 LLM을 공부하고 있어.",
    },
    {
        "role": "assistant",
        "content": "Edge AI와 LLM을 공부하고 있군요.",
    },

    {
        "role": "user",
        "content": "최종 목표는 인터넷 없이 Gemma를 로컬에서 실행하는 거야.",
    },
    {
        "role": "assistant",
        "content": "Gemma를 완전히 로컬 환경에서 실행하는 것이 목표군요.",
    },
]


question = \
"""
사용하는 장치, 공부 중인 분야,
그리고 최종 목표를 모두 정리해줘.
"""

응답 생성 함수도 정의합니다.

In [7]:
def generate_response(messages):
    response = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages,
        max_tokens=100,
        temperature=0.7,
    )

    role = response.choices[0].message.role
    content = response.choices[0].message.content

    return role, content

#### $1)$ Full History

Full History 방식의 Memory는 앞선 실습에서 확인한 방식과 동일합니다.

`client.chat.completions.create`의 `messages`안에 지금까지의 모든 대화 내용을 입력합니다.

여기서 중요한 점은:
1. 모든 대화 기록(Conversation History)을 `messages`에 유지하며 Prompt로 사용
2. 새로운 질문/요청을 `append`하여 Prompt로 Client에 전달
3. 응답도 `messages`에 `append`하여 대화 기록 업데이트
4. 업데이트된 모든 대화 기록을 다시 Prompt로 사용

```python
# 1. 모든 대화 기록(Conversation History)을 messages에 유지하며 Prompt로 사용
messages = [
    {
        "role": ....
        "content": ...
    },
    {
        "role": ...,
        "content": ...
    },
    {
        "role": ...,
        "content": ...
    },
    ...
]


# 2. 새로운 질문/요청을 `append`하여 Prompt로 Client에 전달
messages.append(
    {
        "role": "user",
        "content": "..."
    }
)

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    max_tokens=100,
    temperature=0.7,
)


# 3. 응답도 `messages`에 `append`하여 대화 기록 업데이트
messages.append(
    {
        "role": response.choices[0].message.role,
        "content": response.choices[0].message.content
    }
)


# 4. 업데이트된 모든 대화 기록을 다시 Prompt로 사용
messages.append(
    {
        "role": "user",
        "content": "..."
    }
)

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages,
    max_tokens=100,
    temperature=0.7,
)
```

위에서 정의한 대화 기록과 질문을 사용하여 Full History Memory를 구현해봅시다.

In [13]:
full_messages = deepcopy(history)

full_messages.append(
    {
        "role": "user",
        "content": question
    }
)

role, content = generate_response(full_messages)

full_messages.append(
    {
        "role": role,
        "content": content
    }
)

for idx, msg in enumerate(full_messages):
    print(f"[{idx}]", msg["role"], ":", msg["content"])

[0] user : 나는 Jetson Orin Nano 8GB를 사용하고 있어.
[1] assistant : Jetson Orin Nano 8GB를 사용하고 있군요.
[2] user : 현재 Edge AI와 LLM을 공부하고 있어.
[3] assistant : Edge AI와 LLM을 공부하고 있군요.
[4] user : 최종 목표는 인터넷 없이 Gemma를 로컬에서 실행하는 거야.
[5] assistant : Gemma를 완전히 로컬 환경에서 실행하는 것이 목표군요.
[6] user : 
사용하는 장치, 공부 중인 분야,
그리고 최종 목표를 모두 정리해줘.

[7] assistant : 네, 정리해 드리겠습니다.

*   **사용 장치:** Jetson Orin Nano 8GB
*   **공부 분야:** Edge AI, LLM (Large Language Models)
*   **최종 목표:** 인터넷 연결 없이 Jetson Orin Nano 8GB에서 Gemma를 로컬에서 실행

현재 목표를 달성하기 위해 어떤 어려움을 겪고 계신가요? 혹시 특정 부분에 대한 질문이나 도움이 필요하신 부분이 있다면 알려


#### $2)$ Sliding Window

최근 2개 Turn만 남기는 Sliding Window Memory를 구현해봅시다.

In [19]:
TURNS = 2


recent_history = deque(deepcopy(history), maxlen=TURNS*2)

recent_history.append(
    {
        "role": "user",
        "content": question,
    }
)

role, content = generate_response(list(recent_history))

recent_history.append(
    {
        "role": role,
        "content": content
    }
)

for idx, msg in enumerate(recent_history):
    print(f"[{idx}]", msg["role"], ":", msg["content"])

[0] user : 최종 목표는 인터넷 없이 Gemma를 로컬에서 실행하는 거야.
[1] assistant : Gemma를 완전히 로컬 환경에서 실행하는 것이 목표군요.
[2] user : 
사용하는 장치, 공부 중인 분야,
그리고 최종 목표를 모두 정리해줘.

[3] assistant : 네, 정리해 드리겠습니다.

**1. 장치:**

*   **주 사용 장치:** MacBook Pro (M2 Pro 칩)
*   **추가 고려 장치:** Raspberry Pi 4 (필요한 경우) - 로컬 실행 성능 테스트 및 최적화용

**2. 공부 중인 분야:**

*   **Edge AI:**
    *   TensorFlow Lite, ONNX Runtime 등 Edge AI 프레임워크 학습
    


#### $3)$ Summary Memory

마지막으로, 대화 내용을 요약해서 저장하는 Summary Memory를 구현해봅시다.

In [25]:
TURNS = 2


# 최근 4개 Message
recent_history = deque(deepcopy(history), maxlen=TURNS*2)


# recent_history[0]까지 포함한 이전 대화
old_history = history[:-(TURNS*2 - 1)]


old_history_text = ""

for message in old_history:
    old_history_text += (f"{message['role']}: {message['content']}\n")


summary_prompt = f"""
다음은 사용자와 AI의 이전 대화야.
이후 대화에서 필요할 수 있는 중요한 정보만
짧게 요약해줘.

대화:
{old_history_text}

요약:
"""


summary_messages = [
    {
        "role": "user",
        "content": summary_prompt,
    }
]


_, summary_content = generate_response(summary_messages)


# recent_history의 첫 번째 Message를 이전 대화 요약으로 대체
recent_history[0] = {
    "role": "user",
    "content": f"[이전 대화 요약]\n{summary_content}",
}


# 현재 질문을 추가하여 LLM 응답 생성
messages = list(recent_history)

messages.append(
    {
        "role": "user",
        "content": question,
    }
)

role, content = generate_response(messages)


# 기존 recent_history의 첫 3개를 다시 요약
history_to_summarize = list(recent_history)[:3]

history_to_summarize_text = ""

for message in history_to_summarize:
    history_to_summarize_text += (f"{message['role']}: {message['content']}\n")


summary_prompt = f"""
다음은 사용자와 AI의 이전 대화야.
이후 대화에서 필요할 수 있는 중요한 정보만
짧게 요약해줘.

대화:
{history_to_summarize_text}

요약:
"""


summary_messages = [
    {
        "role": "user",
        "content": summary_prompt,
    }
]

_, summary_content = generate_response(summary_messages)


# 첫 3개 제거
for _ in range(3):
    recent_history.popleft()


# 요약된 내용 추가
recent_history.appendleft(
    {
        "role": "user",
        "content": f"[이전 대화 요약]\n{summary_content}",
    }
)


# 방금 질문과 응답 추가
recent_history.append(
    {
        "role": "user",
        "content": question,
    }
)

recent_history.append(
    {
        "role": role,
        "content": content,
    }
)


# 결과 확인
for idx, msg in enumerate(recent_history):
    print(f"[{idx}]", msg["role"], ":", msg["content"])

[0] user : [이전 대화 요약]
*   사용자: Jetson Orin Nano 8GB 사용, Edge AI & LLM 학습 중
*   목표: 인터넷 없이 Gemma를 로컬에서 실행
[1] assistant : Gemma를 완전히 로컬 환경에서 실행하는 것이 목표군요.
[2] user : 
사용하는 장치, 공부 중인 분야,
그리고 최종 목표를 모두 정리해줘.

[3] assistant : 네, 정리해 드리겠습니다.

*   **장치:** Jetson Orin Nano 8GB
*   **공부 분야:** Edge AI, LLM (Large Language Models)
*   **최종 목표:** 인터넷 연결 없이 Gemma를 로컬에서 실행

혹시 이 목표를 달성하기 위해 어떤 어려움을 겪고 있거나, 어떤 부분에 대한 도움이 필요하신가요?


---

### F. Gemma

**Gemma:**
* Google에서 개발한 경량 LLM 계열
* Transformer 기반의 생성형 언어 모델
* 다양한 Parameter 크기의 모델 제공
* 모델 가중치를 공개하여 로컬 실행 가능
* 제한된 자원의 Edge Device에서도 활용 가능

#### $1)$ 모델 다운로드

기존 실습에서는 API를 통해 원격 환경의 Gemma를 사용했다면, 이번에는 Gemma를 로컬 환경에서 직접 실행하기 위해 모델을 다운로드해보겠습니다.

In [46]:
import os
from dotenv import load_dotenv

from huggingface_hub import hf_hub_download


load_dotenv()

TOKEN = os.getenv("HF_TOKEN")
REPO_ID = "bartowski/google_gemma-4-E2B-it-GGUF"
MODEL_FILE = "google_gemma-4-E2B-it-Q4_K_M.gguf"
MMPROJ_FILE = "mmproj-google_gemma-4-E2B-it-f16.gguf"
MODEL_DIR = "src/models/Gemma4"

In [47]:
print("다운로드 시작")

model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=MODEL_FILE,
    token=TOKEN,
    local_dir=MODEL_DIR,
)

mmproj_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=MMPROJ_FILE,
    token=TOKEN,
    local_dir=MODEL_DIR,
)

print(f"다운로드 완료:\n{model_path}\n{mmproj_path}")

다운로드 시작


google_gemma-4-E2B-it-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 3.46GB            

google_gemma-4-E2B-it-Q4_K_M.gguf: downloading bytes:           |  0.00B            

mmproj-google_gemma-4-E2B-it-f16.gguf: reconstructing file:   0%|          |  0.00B /  986MB            

mmproj-google_gemma-4-E2B-it-f16.gguf: downloading bytes:           |  0.00B            

다운로드 완료:
/home/limjinwoo/vision-LLM/src/models/Gemma4/google_gemma-4-E2B-it-Q4_K_M.gguf
/home/limjinwoo/vision-LLM/src/models/Gemma4/mmproj-google_gemma-4-E2B-it-f16.gguf


#### $2)$ 패키지 설치

`llama-cpp-python`은 `llama.cpp`를 Python에서 사용할 수 있도록 제공하는 라이브러리입니다. 이를 사용하면 GGUF 형식의 LLM을 로컬 환경에서 직접 불러와 추론할 수 있습니다. 또한 CPU뿐 아니라 CUDA를 이용한 GPU 가속도 지원합니다.

Jetson에서는 GPU 가속을 사용하기 위해 CUDA 기반으로 `llama-cpp-python`을 직접 빌드하여 설치해야 합니다. 따라서 CUDA Compiler인 `nvcc`의 경로를 지정하고, `GGML_CUDA` 옵션을 활성화하여 설치합니다.

llama-cpp-python 설치:

```bash
export PATH=/usr/local/cuda/bin:$PATH
CUDACXX=/usr/local/cuda/bin/nvcc CMAKE_ARGS="-DGGML_CUDA=on -DCMAKE_CUDA_COMPILER=/usr/local/cuda/bin/nvcc" \
pip install --no-cache-dir llama-cpp-python
```

In [2]:
from llama_cpp import Llama

`llama_cpp` 패키지 import가 정상적으로 이루어진다면 다음으로 넘어가면 되겠습니다.

#### $3)$ NVIDIA Jetson Linux 업그레이드

로컬에 설치한 Gemma 모델을 불러오기 앞서, Ubuntu 위에서 Jetson 하드웨어를 동작시키기 위해 사용하는 NVIDIA Jetson Linux 버전을 확인합시다.

```bash
cat /etc/nv_tegra_release | head -n 1
```

위 명령어의 결과가 `# R36 (release), REVISION: 4.7`과 같이 출력된다면, Jetson Linux R36.4.7을 사용 중이므로 먼저 R36.5 이상으로 업데이트합니다.

현재 공식적으로 NVIDIA가 확인한 문제로, R36.4.7에서 메모리가 충분히 남아 있음에도 CUDA 메모리 할당 오류가 발생할 수 있으며 해당 문제는 R36.5에서 수정되었습니다.

먼저 현재 NVIDIA repo를 확인합니다.

```bash
cat /etc/apt/sources.list.d/nvidia-l4t-apt-source.list
```

아마 r36.4 계열로 되어 있을 겁니다.

이 파일의 repository 버전을 r36.5로 변경하도록 합시다.

```bash
sudo sed -i 's/r36\.4/r36.5/g' /etc/apt/sources.list.d/nvidia-l4t-apt-source.list
```

파일 수정 후 확인해봅시다.

```bash
cat /etc/apt/sources.list.d/nvidia-l4t-apt-source.list
```

r36.5 계열로 변경되어 있다면 정상적으로 수정이 완료된 상태입니다.

그 다음, 업데이트를 합니다.

```bash
sudo apt update
```

```bash
apt-cache policy nvidia-l4t-core
```

위 명령어에서 다음과 같은 결과가 출력되는 것을 확인하도록 합시다.
```text
nvidia-l4t-core:
  Installed: 36.4.7-20250918154033
  Candidate: 36.5.2-20260716114719
```

Candidate가 36.5라면 업그레이드를 진행합니다.

```bash
sudo apt dist-upgrade
sudo apt install --fix-broken -o Dpkg::Options::="--force-overwrite"
```

완료되면 재부팅하여 다음을 확인합니다.

```bash
cat /etc/nv_tegra_release | head -n 1
```

이제 다음과 같은 결과가 출력되어야 합니다.
```text
# R36 (release), REVISION: 5 ...
```

해당 출력이 확인되었다면, 다음으로 넘어가면 됩니다.

#### $4)$ 로컬 Gemma 모델 로드

GGUF 모델 파일을 불러와서 실제 추론에 사용할 LLM 객체를 생성합니다.

In [3]:
MODEL_PATH = "src/models/Gemma4/google_gemma-4-E2B-it-Q4_K_M.gguf"


llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,   # GPU 가속 사용
    n_ctx=2048,        # Context Window 크기
    n_batch=32,        # 한 번의 decode 요청에서 처리할 수 있는 최대 token 수
    n_ubatch=32,       # GPU/CPU가 한 번에 계산하는 token 묶음의 최대 크기
    verbose=False,
)

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512


#### $5)$ 첫 번째 로컬 텍스트 생성

불러온 로컬 Gemma 모델을 사용하여 텍스트를 생성해봅시다.

In [4]:
response = llm.create_chat_completion(
    messages=[
        {
            "role": "user",
            "content": "LLM이 무엇인지 간단하게 설명해줘.",
        }
    ],
    max_tokens=100,
)


print(response["choices"][0]["message"]["content"])

**LLM(거대 언어 모델, Large Language Model)은** 방대한 양의 텍스트 데이터를 학습하여 **인간과 매우 유사한 언어를 이해하고 생성할 수 있는 인공지능 모델**입니다.

**쉽게 비유하자면:**

* **엄청나게 많은 책을 읽고 공부한 학생**과 같다고 생각할 수 있습니다.
* 이 학생은 세상의 다양한 지식과 언어 패턴을 학습했기


유저 Input을 받아 텍스트를 생성해봅시다.

In [5]:
response = llm.create_chat_completion(
    messages=[
        {
            "role": "user",
            "content": input("무엇을 도와드릴까요?"),
        }
    ],
    max_tokens=100,
)


print(response["choices"][0]["message"]["content"])

정말 힘드시군요. 지금 어떤 일 때문에 그렇게 힘드신가요?

**괜찮으시다면, 조금 더 자세히 이야기해 주시겠어요?** 제가 들어드릴 준비가 되어 있습니다.

* **힘든 감정** (슬픔, 불안, 지침 등)을 표현해 주세요.
* **무엇이 가장 힘든지** (특정 상황, 관계, 걱정 등) 알려주세요.
* **지금 당장 필요한


#### $6)$ 간단한 Memory-Less 챗봇 생성

In [ ]:
while True:
    user_input = input("무엇을 도와드릴까요?")

    if user_input.lower() in ["exit", "quit", "q", "종료"]:
        break

    messages = [
        {
            "role": "user",
            "content": user_input
        }
    ]

    response = llm.create_chat_completion(
        messages=messages,
        max_tokens=300,
    )

    answer = response["choices"][0]["message"]["content"]

    print(answer)

어떤 종류의 것을 원하시나요? 예를 들어, 다음과 같은 것들을 요청하실 수 있습니다.

* **재미있는 이야기:** 짧은 농담이나 흥미로운 이야기
* **정보:** 특정 주제에 대한 사실이나 설명
* **창의적인 글:** 시, 짧은 이야기, 노래 가사 등
* **아이디어:** 새로운 취미, 프로젝트 아이디어 등
* **질문:** 궁금한 것이 있다면 무엇이든 물어보세요.
* **무작위 질문:** 제가 무작위로 질문을 드릴 수도 있습니다.

**지금 바로 제가 무언가를 해 드릴까요? 아니면 특별히 생각하고 계신 것이 있나요? 😊**
오늘 저녁 메뉴를 고르는 데 도움이 필요하신가요? 😊

어떤 종류의 음식을 선호하시나요? 예를 들어, 다음과 같은 정보들을 알려주시면 더 맞춤화된 추천을 해 드릴 수 있어요!

1. **어떤 종류의 음식을 드시고 싶으세요?** (한식, 중식, 일식, 양식, 퓨전 등)
2. **요리하는 데 얼마나 시간을 투자할 수 있나요?** (간단하게, 보통, 시간이 좀 걸려도 괜찮음)
3. **함께 식사하는 사람이 있나요?** (혼자, 가족, 친구 등)
4. **특별히 먹고 싶은 재료나 피하고 싶은 재료가 있나요?**
5. **오늘의 기분이나 상황은 어떤가요?** (매콤한 것, 따뜻하고 편안한 것, 가볍고 건강한 것 등)

---

**💡 만약 아직 정하지 못하셨다면, 몇 가지 추천 옵션을 드려볼게요!**

**🍳 간단하고 빠르게 만들고 싶을 때:**
* **김치볶음밥/참치마요 덮밥:** 실패 없이 만족도가 높아요.
* **파스타 (알리오 올리오 또는 토마토):** 재료가 간단하고 분위기도 좋아요.
* **계란 볶음밥 또는 오믈렛:**
어떤 종류의 저녁 식사를 원하시나요? 몇 가지 질문에 답해주시면 더 좋은 추천을 해 드릴 수 있어요! 😊

**다음 정보들을 알려주시면 맞춤 추천이 가능해요:**

1. **누구와 함께 드시나요?** (혼자, 가족, 친구 등)
2. **요리하는 데 얼마나 시간을 투자할 수 있나요?** (간단하게, 보통, 시간이 좀 걸

#### 최종 코드:

In [1]:
from llama_cpp import Llama


MODEL_PATH = "src/models/Gemma4/google_gemma-4-E2B-it-Q4_K_M.gguf"
CONTEXT_WINDOW = 2048
MAX_TOKENS = 300


llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,
    n_ctx=CONTEXT_WINDOW,
    n_batch=32,
    n_ubatch=32,
    verbose=False,
)


print("Gemma 4 Local Chatbot")
print("종료하려면 exit, quit, q, 종료 중 하나를 입력하세요.\n")


while True:
    user_input = input("무엇을 도와드릴까요?")

    if user_input.lower() in ["exit", "quit", "q", "종료"]:
        break

    messages = [
        {
            "role": "user",
            "content": user_input
        }
    ]

    response = llm.create_chat_completion(
        messages=messages,
        max_tokens=MAX_TOKENS,
    )

    answer = response["choices"][0]["message"]["content"]

    print(answer)

llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512
llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 512


Gemma 4 Local Chatbot
종료하려면 exit, quit, q, 종료 중 하나를 입력하세요.

That looks like a random string of letters. Is there something you'd like me to do with it? For example, are you:

* **Trying to type something specific?**
* **Testing the system?**
* **Looking for a definition or meaning?**
* **Experiencing a technical issue?**

Let me know how I can help! 😊
"d" is a single letter.

To give you a more helpful answer, could you tell me what you'd like to know about "d"? For example, are you interested in:

* **Its alphabetical position?** (It's the 4th letter of the English alphabet.)
* **Its use in words?** (e.g., dog, day, dream)
* **Its mathematical representation?** (as a variable)
* **A specific context?** (like a chemical symbol, a programming variable, etc.)

**Please provide more context!** 😊
네, 알겠습니다. **'d'는 필요 없다**는 말씀이시군요.

혹시 다른 궁금한 점이나 도움이 필요하신 것이 있으신가요? 😊


KeyboardInterrupt: Interrupted by user

이것으로 **『LLM과 Gemma』** 섹션을 마무리합니다.

---

## <center>< Section Project ></center>

본 섹션에서 배운 내용을 토대로 프로젝트를 진행합니다.<br>
로컬 Gemma 모델을 사용하여 Summary Memory 기반 챗봇을 만들어봅시다.

In [2]:
# TODO: Section 5 "LLM과 Gemma" Project

---
---

<br><br><div style="text-align: right; color: gray; font-style: italic;">
© 2026 김규래 (Kyu Rae Kim). All rights reserved.&emsp;<br><br>
This material is provided solely for the intended instructional purpose.&emsp;<br>
Redistribution, reproduction, modification, adaptation, or reuse of this material in any form without prior written permission from the copyright holder is prohibited.&emsp;
</div>